# Qwen 3.6 27B - Colab Host with vLLM + Tool Calling

⚠️ **READ THIS FIRST**

This notebook loads **Qwen/Qwen3.6-27B** (full BF16 weights, ~54GB) via **vLLM** on a Colab A100.

**VRAM Reality Check:**
- This model needs ~54GB just for the weights at BF16
- You MUST have an **A100 80GB**. A100 40GB, V100, T4, or L4 will **OOM immediately**
- Even on 80GB, context length is limited to ~4K-8K tokens (official default is 262K)

**Why vLLM?** vLLM's `--tool-call-parser qwen3_coder` converts Qwen's XML-style tool output into proper OpenAI `tool_calls` objects, so Pi can execute tools.

## 1. Check GPU & VRAM (MUST be A100 80GB)

In [ ]:
#@title 1. Check GPU
import subprocess

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],
    capture_output=True, text=True
)
if result.returncode != 0:
    print('No GPU found. Enable GPU in Runtime > Change runtime type.')
    raise SystemExit

name, total, free = [x.strip() for x in result.stdout.strip().split(',')]
total_mb = int(total.split()[0])
free_mb = int(free.split()[0])

print(f'GPU: {name}')
print(f'Total VRAM: {total_mb}MB | Free: {free_mb}MB')

if 'A100' not in name or total_mb < 75000:
    print('\n❌ FATAL: You do not have an A100 80GB GPU.')
    print(f'   You have: {name} with {total_mb}MB total')
    print('   Qwen3.6-27B at BF16 needs ~54GB for weights alone.')
    print('   Stop here and upgrade to Colab Pro+ (A100 80GB),')
    print('   or use the GGUF notebook instead.')
    raise SystemExit('Insufficient GPU memory')

if free_mb < 60000:
    print('\n⚠️ WARNING: Less than 60GB free. May OOM during load.')
    print('   Try Runtime > Disconnect and delete runtime, then re-run.')
else:
    print('\n✅ A100 80GB detected. Proceeding...')


## 2. Install vLLM (>=0.19.0 required)

Colab sometimes lacks the latest pre-built wheel. If pip install fails, we try building from source.

In [ ]:
#@title 2. Install vLLM
import os, sys, subprocess

print('Checking current vLLM...')
try:
    import vllm
    current = vllm.__version__
    print(f'Current vLLM: {current}')
    major, minor = map(int, current.split('.')[:2])
    if major == 0 and minor < 19:
        print('Version too old. Upgrading...')
        os.system('pip install --upgrade "vllm>=0.19.0"')
    else:
        print('vLLM version OK.')
except ImportError:
    print('vLLM not found. Installing...')
    print('This may take 5-15 minutes.')
    ret = os.system('pip install "vllm>=0.19.0" 2>&1')
    if ret != 0:
        print('Standard install failed. Trying CUDA 12.1 wheel...')
        os.system('pip install "vllm>=0.19.0" --extra-index-url https://download.pytorch.org/whl/cu121')

try:
    import importlib
    importlib.reload(vllm) if 'vllm' in sys.modules else __import__('vllm')
    print(f'vLLM version: {vllm.__version__}')
except Exception as e:
    print(f'ERROR: {e}')
    print('\nIf install keeps failing, you may need to use the GGUF notebook instead.')
    raise


## 3. Authenticate with HuggingFace

**Qwen3.6-27B requires accepting the model license.**

1. Go to https://huggingface.co/Qwen/Qwen3.6-27B and click **Accept License**
2. Get your token from https://huggingface.co/settings/tokens
3. Paste it below and run this cell

In [ ]:
#@title 3. HuggingFace Login
from huggingface_hub import login, whoami

HF_TOKEN = ""  #@param {type:"string"}

if HF_TOKEN:
    login(token=HF_TOKEN)
    try:
        info = whoami()
        print(f'Logged in as: {info.get("name", "unknown")}')
    except Exception:
        print('Login succeeded but whoami failed (token may be read-only).')
else:
    print('Please set HF_TOKEN above.')
    print('Get one at https://huggingface.co/settings/tokens')
    raise SystemExit('No HF token provided')


## 4. Start vLLM OpenAI Server

**Correct flags for Qwen3.6 (from official docs):**
- `--tool-call-parser qwen3_coder` — the ONLY parser that works for Qwen3.6 tool calling
- `--reasoning-parser qwen3` — strips Qwen's `<thinking>...</thinking>` blocks
- `--language-model-only` — skips the vision encoder, saving ~5-10GB VRAM
- `--max-model-len 8192` — conservative; increase only if you have VRAM headroom
- `--gpu-memory-utilization 0.90` — leave headroom for system processes

**Gotcha:** vLLM downloads the full model (~55GB) on first run. This takes 10-20 minutes.

In [ ]:
#@title 4. Start vLLM Server (OpenAI-compatible)
import subprocess, time, urllib.request, json, os

PORT = 8000
MAX_MODEL_LEN = 8192  #@param {type:"integer"}

# Context length tradeoff on A100 80GB with 27B BF16 weights (~54GB):
#   4096  -> safest, plenty of headroom
#   8192  -> good balance, usually fits (DEFAULT)
#   16384 -> tight, may OOM depending on batch size
#   32768 -> very likely to OOM; needs multi-GPU in practice
# The model's official default is 262K tokens, but that needs 8x GPUs.

if MAX_MODEL_LEN > 16384:
    print(f'⚠️  WARNING: max_model_len={MAX_MODEL_LEN} will likely OOM on a single A100 80GB.')
    print('   27B BF16 weights alone use ~54GB. Only ~18GB remains for KV cache.')
    print('   Recommended: 4096 (safe) or 8192 (balanced).')

# Kill any previous server on the same port
!lsof -ti :{PORT} | xargs kill -9 2>/dev/null || true

stdout_log = open("/content/server_stdout.log", "w")
stderr_log = open("/content/server_stderr.log", "w")

server_proc = subprocess.Popen(
    [
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", "Qwen/Qwen3.6-27B",
        "--dtype", "bfloat16",
        "--max-model-len", str(MAX_MODEL_LEN),
        "--gpu-memory-utilization", "0.90",
        "--language-model-only",
        "--enable-auto-tool-choice",
        "--tool-call-parser", "qwen3_coder",
        "--reasoning-parser", "qwen3",
        "--port", str(PORT),
        "--host", "0.0.0.0",
    ],
    stdout=stdout_log,
    stderr=stderr_log,
)

print(f'Server starting on port {PORT} (PID: {server_proc.pid})...')
print('Downloading + loading model (10-20 min on first run)...')

for i in range(240):  # up to 20 minutes
    time.sleep(5)
    if server_proc.poll() is not None:
        print(f'\n❌ Server process exited with code {server_proc.returncode}!')
        print('Last 30 lines of stderr:')
        !tail -30 /content/server_stderr.log 2>/dev/null
        break
    if i % 6 == 5:
        print(f'  Still loading... ({(i+1)*5}s)', flush=True)
    try:
        req = urllib.request.Request(f'http://localhost:{PORT}/v1/models', method='GET')
        resp = urllib.request.urlopen(req, timeout=5)
        if resp.status == 200:
            data = json.loads(resp.read())
            model_id = data['data'][0]['id'] if data.get('data') else 'unknown'
            print(f'\n✅ Server is healthy! Model: {model_id}')
            break
    except Exception:
        pass
else:
    print('\n⚠️  Server not ready after 20 minutes.')
    !tail -30 /content/server_stderr.log 2>/dev/null


## 5. Install & Start Cloudflare Tunnel

In [ ]:
#@title 5. Cloudflare Tunnel
import subprocess, time

!curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

PORT = 8000
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://localhost:{PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

print('Starting tunnel...')

TUNNEL_URL = None
timeout_secs = 30
start = time.time()

while time.time() - start < timeout_secs:
    line = tunnel_proc.stderr.readline().decode(errors='replace')
    if not line and tunnel_proc.poll() is not None:
        print(f'Tunnel process exited unexpectedly with code {tunnel_proc.returncode}')
        break
    if 'https://' in line and '.trycloudflare.com' in line:
        for part in line.split():
            if part.startswith('https://') and '.trycloudflare.com' in part:
                TUNNEL_URL = part.rstrip('/')
                break
    if TUNNEL_URL:
        break

if TUNNEL_URL:
    print(f'\n{"="*60}')
    print(f'YOUR TUNNEL URL: {TUNNEL_URL}')
    print(f'{"="*60}')
    print(f'\nRun locally:')
    print(f"  update-tunnel {TUNNEL_URL}/v1")
else:
    print('\nTunnel URL not found within 30s.')


## 6. Quick Test (Tool Calling + Reasoning)

This sends a tool request and checks if vLLM returns **proper OpenAI `tool_calls`** (not raw XML).

In [ ]:
#@title 6. Test Tool Calling
import json, urllib.request

PORT = 8000
MODEL = "Qwen/Qwen3.6-27B"

payload = {
    "model": MODEL,
    "messages": [{"role": "user", "content": "List files in /content using the bash tool"}],
    "tools": [
        {
            "type": "function",
            "function": {
                "name": "bash",
                "description": "Run a shell command",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "command": {"type": "string"}
                    },
                    "required": ["command"]
                }
            }
        }
    ],
    "tool_choice": "auto",
    "stream": False
}

req = urllib.request.Request(
    f'http://localhost:{PORT}/v1/chat/completions',
    data=json.dumps(payload).encode(),
    headers={'Content-Type': 'application/json'},
    method='POST'
)

try:
    resp = urllib.request.urlopen(req, timeout=60)
    data = json.loads(resp.read())
    msg = data['choices'][0]['message']
    print('=== Response ===')
    print(json.dumps(msg, indent=2))
    print()
    if msg.get('tool_calls'):
        print(f'✅ SUCCESS: {len(msg["tool_calls"])} tool_call(s) detected!')
        for tc in msg['tool_calls']:
            print(f'   Function: {tc["function"]["name"]}')
            print(f'   Args: {tc["function"]["arguments"]}')
        print('\nPi will now be able to execute tools correctly.')
    else:
        print('❌ FAILURE: No tool_calls in response.')
        print('   Content:', msg.get('content', '(empty)')[:500])
        print('\nThe model returned raw text instead of OpenAI tool_calls.')
        print('   Check that --tool-call-parser qwen3_coder is set.')
except Exception as e:
    print(f'ERROR: {e}')


## 7. Debug

In [ ]:
#@title 7. Debug
import subprocess

print('=== Server Process ===')
if 'server_proc' in globals():
    print(f'PID: {server_proc.pid}, Running: {server_proc.poll() is None}')
    if server_proc.poll() is not None:
        print(f'Exit code: {server_proc.returncode}')
else:
    print('No server_proc variable found. Has Cell 4 been run?')

print('\n=== Last 40 lines of server stderr ===')
!tail -40 /content/server_stderr.log 2>/dev/null || echo 'No log file found'

print('\n=== Last 20 lines of server stdout ===')
!tail -20 /content/server_stdout.log 2>/dev/null || echo 'No log file found'

print('\n=== GPU Memory ===')
!nvidia-smi --query-gpu=memory.used,memory.free --format=csv,noheader

print('\n=== Port 8000 Check ===')
!lsof -i :8000 2>/dev/null || echo 'Nothing listening on port 8000'

print('\n=== Tunnel Process ===')
if 'tunnel_proc' in globals():
    print(f'PID: {tunnel_proc.pid}, Running: {tunnel_proc.poll() is None}')
else:
    print('No tunnel_proc variable found. Has Cell 5 been run?')

if 'TUNNEL_URL' in globals() and TUNNEL_URL:
    print(f'\nTunnel URL: {TUNNEL_URL}')
else:
    print('\nNo tunnel URL found.')


## Local Usage

After running all cells, copy the tunnel URL from Cell 5 output.

```bash
update-tunnel https://xxx-yyy-zzz.trycloudflare.com/v1
pi --provider colab-qwen --model default
```

**Note:** Cloudflare assigns a new URL every runtime restart. Update `update-tunnel` each time.